In [1]:
import os
import re
import time
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from scipy import stats
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import ContextualCompressionRetriever, EnsembleRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from dotenv import load_dotenv

load_dotenv()

MODEL = "gpt-4o-mini"

llm = ChatOpenAI(model=MODEL)
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [3]:
documents = [
    Document(page_content="트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.", metadata={"id": "d1"}),
    Document(page_content="BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.", metadata={"id": "d2"}),
    Document(page_content="GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.", metadata={"id": "d3"}),
    Document(page_content="RAG는 검색 증강 생성 기법으로 외부 지식을 LLM에 결합하여 할루시네이션을 줄입니다.", metadata={"id": "d4"}),
    Document(page_content="벡터 데이터베이스는 임베딩 벡터를 저장하고 유사도 기반 검색을 수행합니다. FAISS, Pinecone 등이 있습니다.", metadata={"id": "d5"}),
    Document(page_content="파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.", metadata={"id": "d6"}),
    Document(page_content="프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.", metadata={"id": "d7"}),
    Document(page_content="토큰화는 텍스트를 모델이 처리할 수 있는 단위로 분할하는 과정입니다. BPE, WordPiece 등이 사용됩니다.", metadata={"id": "d8"}),
]

In [23]:
#re-ranking
vectorstore = FAISS.from_documents(documents, embeddings_model)
bm_25_retriever = BM25Retriever.from_documents(documents, k=5)

In [26]:
doc_embeddings = {}

def get_embedding(text):
    return np.array(embeddings_model.embed_query(text))

for doc in doc_embeddings:
    doc_embeddings[doc.metadata['id']] = get_embedding(doc.page_content)

In [2]:
import numpy as np

class AdaptiveFilter:
    @staticmethod
    def scored_docs(scored_docs):
        scores = [s for _, s in scored_docs]
        std = np.std(scores)
        score_range = max(scores) - min(scores)

        if std > 0.2:
            return ScoreFilter.score_gap(scored_docs, 2)
        elif score_range < 0.1:
            return ScoreFilter.fixed_threshold(scored_docs, 0.5)
        else:
            return ScoreFilter.dynamic_threshold(scored_docs, std_factor=1)


AdaptiveFilter.scored_docs(results)

NameError: name 'results' is not defined

In [9]:
document = "BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다. 트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다. GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다."
document.split('')

ValueError: empty separator

In [15]:
# 하네스 엔지니어링 : compress 전략 100k -> 압축해라, 턴이 20턴정도 끝나면 압축해라
# context compression
# retriever : Embedding, Bm25, Hybrid, MMR
# MMR(Maximun Marginal Relevance)

# 정규표현식
import re

def extractive_compress(query, document, max_sentences=3):
    sentences = re.split(r'[.!?]\s*', document)

    if not sentences:
        return document

    query_terms = set(query.lower().split())
    scored = []

    for sent in sentences:
        sent_terms = set(sent.lower().split())
        overlap = len(query_terms & sent_terms)
        scored.append((sent, overlap))

    # 점수 기준 정렬
    scored.sort(key=lambda x: x[1], reverse=True)

    # 상위 문장 선택
    selected = [s for s, _ in scored[:max_sentences] if s.strip()]

    return ". ".join(selected) + "."

In [16]:
long_doc = "트랜스포머는 2017년 구글이 발표한 아키텍처입니다. Self-Attention 메커니즘이 핵심입니다. 이전의 RNN, LSTM과 달리 병렬 처리가 가능합니다. BERT와 GPT 모두 트랜스포머를 기반으로 합니다. 자연어 처리뿐 아니라 컴퓨터 비전에서도 활용됩니다"
query = "트랜스포머와 BERT의 관계"

compressed = extractive_compress(query, long_doc, max_sentences=3)
compressed

'트랜스포머는 2017년 구글이 발표한 아키텍처입니다. Self-Attention 메커니즘이 핵심입니다. 이전의 RNN, LSTM과 달리 병렬 처리가 가능합니다.'

In [17]:
compress_chain = ChatPromptTemplate.from_messages([
    ("system", "당신은 문서 요약 전문가입니다."),
    ("human", """다음 문서에서 쿼리에 답하는데 필요한 핵심 정보만 추출하세요.
    불필요한 내용은 제거하고, {max_tokens}자 이내로 압축하세요.

    쿼리 : {query}
    문서 : {document}

    압축결과 :
    """)
]) | llm | StrOutputParser()

In [18]:
def llm_compress(query, document, max_tokens=100):
    return compress_chain.invoke({
        'query' : query,
        'document' : document,
        'max_tokens' : max_tokens
    })

In [19]:
compressed_llm = llm_compress(query, long_doc)

In [20]:
compressed_llm

'트랜스포머는 2017년 구글이 발표한 아키텍처로, BERT와 GPT의 기반입니다.'

In [24]:
compressor = LLMChainExtractor.from_llm(llm)
compressor_retriever = ContextualCompressionRetriever(base_compressor=compressor,
                                                      base_retriever=vectorstore.as_retriever(search_kwargs={'k' : 3}))

In [25]:
compressed_docs = compressor_retriever.invoke(query)
compressed_docs

[Document(metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
 Document(metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.')]

In [27]:
# 압축 품질 평가 : 키워드 보존유르, 의미 보존도
def evaluate_compression(original, compressed, query):
    # 1. 압축율
    ratio = len(compressed) / len(original)

    # 2. 키워드 보존율
    query_terms = set(query.lower().split())
    orig_terms = set(original.lower().split())
    comp_terms = set(compressed.lower().split())

    orig_query_terms = query_terms & orig_terms
    comp_query_terms = query_terms & comp_terms
    keyword_covarage = len(comp_query_terms) / len(orig_query_terms) if orig_query_terms else 1.0

    # 3. 의미 보존도
    original_emb = np.array(embeddings_model.embed_query(original))
    compressed_emb = get_embedding(compressed)

    senmantic_similarity = np.dot(original_emb, compressed_emb) / (np.linalg.norm(original_emb) * np.linalg.norm(compressed_emb))
    
    return {
        'compression_ratio' : ratio,
        'keyword_covarage' : keyword_covarage,
        'senmantic_similarity' : senmantic_similarity
    }

In [28]:
evaluate_compression(long_doc, compressed_llm, query)

{'compression_ratio': 0.30612244897959184,
 'keyword_covarage': 1.0,
 'senmantic_similarity': np.float64(0.7993876782400248)}

In [29]:
def cosine_sim(original_emb, compressed_emb):
    return np.dot(original_emb, compressed_emb) / (np.linalg.norm(original_emb) * np.linalg.norm(compressed_emb))

In [35]:
import re

def mmr_compress(query, document, max_sentences=3, param=0.5):
    sentences = re.split(r'[.!?]\s*', document)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]

    if len(sentences) <= max_sentences:
        return '. '.join(sentences) + '.'

    query_emb = get_embedding(query)
    sent_emb = [get_embedding(s) for s in sentences]

    selected = []
    remaining = list(range(len(sentences)))

    for _ in range(max_sentences):
        best_score = -float('inf')
        best_idx = -1

        for idx in remaining:
            relevance = cosine_sim(query_emb, sent_emb[idx])

            if selected:
                max_sim = max(cosine_sim(sent_emb[idx], sent_emb[s]) for s in selected)
            else:
                max_sim = 0.0

            mmr = param * relevance - (1 - param) * max_sim

            if mmr > best_score:
                best_score = mmr
                best_idx = idx

        selected.append(best_idx)
        remaining.remove(best_idx)

    return '. '.join(sentences[i] for i in sorted(selected)) + '.'

In [36]:
mmr_compress(query, long_doc, max_sentences=2, param=0.7)

'트랜스포머는 2017년 구글이 발표한 아키텍처입니다. BERT와 GPT 모두 트랜스포머를 기반으로 합니다.'

In [37]:
def average_precision(retrieved_ids, relevant_ids):
    relevant_set = set(relevant_ids)
    hits = 0
    sum_precision = 0.0

    for i, doc_id in enumerate(retrieved_ids):
        if doc_id in relevant_set:
            hit += 1
            precision_at_i = hits / (i + 1)
            sum_precision += precision_at_i

    return sum_precision / len(relevant_set) if relevant_set else 0.0

In [38]:
def mean_average_precision(query_results):
    # query_results : {query : (retrieved_ids, relevant_ids)}
    aps = []
    for query, (retrieved, relevant) in query_results.items():
        ap = average_precision(retrieved, relevant)
        aps.append(ap)

    map_score = np.mean(aps)
    return map_score

In [39]:
relevant = {'d1', 'd2', 'd3'}
orig_order = [doc.metadata['id'] for doc, _ in results]
orig_order

NameError: name 'results' is not defined

In [40]:
def intra_list_similarity(doc_ids, doc_embeddings, top_k=5):
    ids = doc_ids[:top_k]
    embs = [doc_embeddings[did] for did in ids if did in doc_embeddings]

    if len(embs) < 2:
        return 0.0

    sims = []
    for i in range(len(embs)):
        for j in range(i+1, len(embs)):
            sims.append(cosine_sim(embs[i], embs[j]))

    return np.mean(sims)